In [10]:
import timm
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)

In [11]:
import os
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
import torch 
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm 
import torch.nn as nn

class RealFakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        # Expecting subfolders 'real/' and 'fake/' inside root_dir
        for label_str, label in [("real", 0), ("fake", 1)]:
            folder = Path(root_dir) / label_str
            if not folder.exists():
                continue
            for file in folder.glob("*.png"):
                self.samples.append((file, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


    def collate_fn(self, batch):
        images, labels = zip(*batch)  # unzip list of tuples
        images = torch.stack(images)  # stack image tensors into a single batch tensor
        labels = torch.tensor(labels) # convert labels to a tensor
        return {"image": images, "label":labels }


        

In [12]:
train_path = "dataset/train/"
test_path = "dataset/test/"
validation_path = "dataset/validation/"

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [13]:
train_dataset = RealFakeDataset(train_path, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, collate_fn=train_dataset.collate_fn)

test_dataset = RealFakeDataset(test_path, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=4, collate_fn=test_dataset.collate_fn)

validation_dataset = RealFakeDataset(validation_path, transform=transform)
validation_dataloader = DataLoader(validation_dataset, batch_size=32, shuffle=True, num_workers=4, collate_fn=validation_dataset.collate_fn)

In [14]:
for i, elem in enumerate(train_dataloader):
    if(i==1):
        break
    print(elem["image"].shape)

torch.Size([32, 3, 256, 256])


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_ce = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), 0.001)


In [16]:
for param in model.parameters():
    param.requires_grad = True

In [17]:
model.to(device)
for epoch in range(1):
    model.train()
    for batch in train_dataloader:
        images, labels = batch["image"], batch["label"]
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_ce(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{1}], Loss: {loss.item():.4f}")


Epoch [1/1], Loss: 0.2159


In [21]:
def report(model, test_loader, loss):
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
    import matplotlib.pyplot as plt
    model = model.to(device)
    model.eval()
    all_predictions = []
    all_targets = []

    total_loss = 0.0
    for batch in tqdm(test_loader):
        input, target = batch["image"], batch["label"]

        input = input.to(device)
        target = target.to(device)

        output = model(input)

        l = loss(output, target)
        total_loss += l.item()

        # Get predictions
        preds = torch.argmax(output, dim=1)  # Adjust if it's binary or multilabel

        all_predictions.extend(preds.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

    # Final cleanup after evaluation
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(all_targets, all_predictions, digits=4))  # Adjust labels if needed

    # Plot confusion matrix
    label_names = ["REAL", "FALSE"]
    
    cm = confusion_matrix(all_targets, all_predictions)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()

In [22]:
report(model, test_dataloader, loss_ce)

  0%|          | 0/422 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 11.58 GiB of which 110.19 MiB is free. Including non-PyTorch memory, this process has 9.98 GiB memory in use. Of the allocated memory 9.49 GiB is allocated by PyTorch, and 260.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)